# Colour-jitter test: does colour augmentation help on museum images?

Before you start: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About **1 hour** on a T4 (3 seeds, about 20 minutes each).

**The only change from the baseline:** WikiArt *training* images get `ColorJitter(brightness=0.2, saturation=0.2)` (random brightness and saturation, up to ±20%). Everything else is the baseline ResNet-50 run (`src/train.py`): same artist-disjoint split, 12 epochs, batch 64, lr 1e-4, class-weighted loss with label smoothing 0.1, best epoch chosen on WikiArt val, seeds 1, 2, 3. Labels are unchanged. WikiArt val/test and the Met and Rijksmuseum images are **not** jittered.

Main numbers are for **painters the model never saw in WikiArt** (unseen artists), as in RQ1. Baseline (3 seeds, balanced accuracy): WikiArt test 61.1, Met 61.0 (unseen 57.5), Rijks 58.0 (unseen 58.2).

**If Colab disconnects:** just **Run all** again. Each finished seed is saved to Drive and skipped next time.

Results go to `wikiart_runs/colorjitter/` in the same Drive folder as `colab_data.zip`. The last cell prints one table to paste back.

## 1. Check for a T4 GPU, connect Drive, unpack the data (about 3 minutes)

In [ ]:
SEEDS = [1, 2, 3]
import torch, os, time, glob
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Go to Runtime -> Change runtime type -> pick T4 GPU -> Save, then Runtime -> Run all again. '
                     '(If T4 is greyed out, your free GPU time is used up for today: try again later or use your other account.)')
GPU = torch.cuda.get_device_name(0)
print('GPU:', GPU)
if 'T4' not in GPU:
    print('Note: this is not a T4. The run still works (the baseline used a T4); training will just be faster.')

from google.colab import drive
drive.mount('/content/drive')
# Same Drive layout as improve_accuracy.ipynb. On the second account the shared folder is a shortcut
# called wikiart-to-museum inside My Drive, so look there first.
CANDIDATES = ['/content/drive/MyDrive/wikiart-to-museum', '/content/drive/MyDrive']
BASE = next((c for c in CANDIDATES if os.path.exists(f'{c}/colab_data.zip')), None)
if BASE is None:
    raise SystemExit('colab_data.zip not found in My Drive or My Drive/wikiart-to-museum. '
                     'Add the shared wikiart-to-museum folder to My Drive (right-click -> Organize -> Add shortcut), then Run all again.')
OUT = f'{BASE}/wikiart_runs/colorjitter'
os.makedirs(OUT, exist_ok=True)
print('Data :', f'{BASE}/colab_data.zip')
print('Saves:', OUT)

if not os.path.exists('/content/data/final/wikiart_split.csv'):
    !cp "{BASE}/colab_data.zip" /content/
    !cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
assert os.path.exists('/content/data/final/museum_test.csv'), 'unzip failed: data/final/museum_test.csv missing'

## 2. Get the training script with the one-line change
Uses the repository's `src/train.py`, which has a `--color-jitter` switch. Its only effect is this line in the *training* transforms:
```python
train.append(transforms.ColorJitter(brightness=0.2, saturation=0.2))
```
If GitHub can't be reached, the same line is added to the `train.py` inside `colab_data.zip` instead.

In [ ]:
import urllib.request, re
URL = 'https://raw.githubusercontent.com/abdull254/wikiart-to-museum/main/src/train.py'
JITTER = 'transforms.ColorJitter(brightness=0.2, saturation=0.2)'
try:
    src = urllib.request.urlopen(URL, timeout=30).read().decode()
    assert '--color-jitter' in src and JITTER in src
    print('Using src/train.py from GitHub')
except Exception as e:
    print('GitHub not reachable or file changed (', e, '): patching the train.py from colab_data.zip')
    src = open('/content/train.py').read()
    if '--color-jitter' not in src:
        anchor = 'transforms.RandomHorizontalFlip()]'
        assert src.count(anchor) == 1, 'could not find the training transforms in train.py'
        src = src.replace(anchor, anchor + '\n    if args.color_jitter:\n        train.append(' + JITTER + ')')
        src = src.replace('    args = ap.parse_args()',
                          '    ap.add_argument("--color-jitter", action="store_true")\n    args = ap.parse_args()', 1)
open('/content/train_cj.py', 'w').write(src)

# Check: the jitter is in the training transforms and nowhere else
import importlib.util, argparse
spec = importlib.util.spec_from_file_location('train_cj', '/content/train_cj.py'); T = importlib.util.module_from_spec(spec); spec.loader.exec_module(T)
tr, te = T.make_transforms(argparse.Namespace(grayscale_aug=0.0, color_jitter=True))
print('train transforms:', [type(t).__name__ for t in tr.transforms])
print('test  transforms:', [type(t).__name__ for t in te.transforms])
assert any(type(t).__name__ == 'ColorJitter' for t in tr.transforms)
assert not any(type(t).__name__ == 'ColorJitter' for t in te.transforms)

## 3. Train 3 seeds (about 20 minutes each on a T4)
Each seed is copied to Drive only when it has finished (metrics + a full checkpoint). Seeds already on Drive are skipped, so after a disconnect just **Run all** again.

In [ ]:
import shutil, json
def finished(d):
    return os.path.exists(f'{d}/metrics.json') and os.path.exists(f'{d}/best.pt') and os.path.getsize(f'{d}/best.pt') > 10_000_000

%cd /content
for SEED in SEEDS:
    name = f'resnet50_cj_seed{SEED}'
    dst = f'{OUT}/{name}'
    if finished(dst):
        print(f'seed {SEED}: already on Drive, skipping'); continue
    t0 = time.time()
    print(f'\n===== seed {SEED}: training with colour jitter =====', flush=True)
    if os.path.exists(f'/content/runs/{name}'):
        shutil.rmtree(f'/content/runs/{name}')   # never reuse a half-finished local run
    !python /content/train_cj.py --seed {SEED} --color-jitter --eval-museums --workers 2 --name {name}
    src_dir = f'/content/runs/{name}'
    if not finished(src_dir):
        raise SystemExit(f'seed {SEED} did not finish. Run all again to retry it.')
    if os.path.exists(dst):
        shutil.move(dst, dst + time.strftime('_old_%Y%m%d_%H%M%S'))
    shutil.copytree(src_dir, dst)
    assert finished(dst), 'copy to Drive failed: check Drive storage space'
    print(f'seed {SEED}: saved to Drive in {(time.time() - t0) / 60:.0f} min')
print('\nAll seeds done:', sorted(os.listdir(OUT)))

## 4. Compare with the baseline
Baseline predictions are the 3 committed baseline seeds in the repository (`results/runs/resnet50_seed1-3`), on the same test images. Seeds are paired (baseline seed 1 vs jitter seed 1, and so on).

* **Paired t-test across seeds**: the 3 seed-by-seed differences (only 3 pairs, so low power).
* **Painter-level paired bootstrap**: resample painters (not single paintings) 2,000 times, same resample for both models; 95% interval and two-sided p-value for the mean difference.
* **McNemar's exact test** per seed: on the same paintings, how many only one of the two models gets right.

In [ ]:
import numpy as np, pandas as pd
from math import comb
from scipy import stats
COMMIT = '5d10b4b'   # baseline runs as committed in the repo
BL = '/content/baseline'
for s in SEEDS:
    os.makedirs(f'{BL}/resnet50_seed{s}', exist_ok=True)
    for k in ('wikiart_test', 'met', 'rijks'):
        f = f'{BL}/resnet50_seed{s}/preds_{k}.csv'
        if not os.path.exists(f):
            urllib.request.urlretrieve(f'https://raw.githubusercontent.com/abdull254/wikiart-to-museum/{COMMIT}/results/runs/resnet50_seed{s}/preds_{k}.csv', f)

# painter of every test painting (for the painter-level bootstrap)
split = pd.read_csv('/content/data/final/wikiart_split.csv')
museum = pd.read_csv('/content/data/final/museum_test.csv')
ARTIST = dict(zip(split['wikiart_idx'].astype(str), 'w:' + split['artist_key'].astype(str)))
ARTIST.update(dict(zip(museum['museum'] + ':' + museum['id'].astype(str), 'm:' + museum['artist'].fillna('unknown').astype(str))))

def load(run_dirs, k):
    fr = [pd.read_csv(f'{d}/preds_{k}.csv', dtype={'id': str}).sort_values('id').reset_index(drop=True) for d in run_dirs]
    return fr
base_runs = [f'{BL}/resnet50_seed{s}' for s in SEEDS]
cj_runs = [f'{OUT}/resnet50_cj_seed{s}' for s in SEEDS]
CLASSES = sorted(c[2:] for c in pd.read_csv(f'{base_runs[0]}/preds_met.csv', nrows=1).columns if c.startswith('p_'))
C = len(CLASSES)

def bal_acc(y, p):  return float(np.mean([(p[y == c] == c).mean() for c in range(C) if (y == c).any()]))
def macro_f1(y, p):
    f = [2 * ((p == c) & (y == c)).sum() / ((p == c).sum() + (y == c).sum()) for c in range(C) if (y == c).any()]
    return float(np.mean(f))

SETS = [('wikiart_test', None), ('met', 'unseen'), ('rijks', 'unseen'), ('met', None), ('rijks', None)]
def label(k, g): return f'{k} (unseen)' if g else (k if k == 'wikiart_test' else f'{k} (all)')

def get(k, g):
    B, J = load(base_runs, k), load(cj_runs, k)
    assert all((b['id'].values == B[0]['id'].values).all() for b in B + J), f'{k}: test images differ between runs'
    keep = np.ones(len(B[0]), bool) if g is None else (B[0]['artist_group'].values == g)
    idx = {c: i for i, c in enumerate(CLASSES)}
    y = B[0]['true'].map(idx).values[keep]
    PB = np.stack([b['pred'].map(idx).values[keep] for b in B])
    PJ = np.stack([j['pred'].map(idx).values[keep] for j in J])
    art = B[0]['id'].map(ARTIST).fillna('unknown').values[keep]
    return y, PB, PJ, art

def boot_painters(y, PB, PJ, art, n_boot=2000, seed=0):
    # Resample painters with replacement; scores from per-painter count tables (fast).
    a_ids, a = np.unique(art, return_inverse=True); A = len(a_ids)
    N = np.zeros((A, C)); np.add.at(N, (a, y), 1)
    def tables(P):
        K = np.zeros((len(P), A, C)); Q = np.zeros((len(P), A, C))
        for s, p in enumerate(P):
            np.add.at(K[s], (a, y), (p == y)); np.add.at(Q[s], (a, p), 1)
        return K, Q
    KB, QB = tables(PB); KJ, QJ = tables(PJ)
    W = np.random.default_rng(seed).multinomial(A, np.ones(A) / A, size=n_boot).astype(float)
    n = W @ N
    def scores(K, Q):
        ba, f1 = [], []
        with np.errstate(invalid='ignore', divide='ignore'):
            for s in range(len(K)):
                tp, pc = W @ K[s], W @ Q[s]
                ba.append(np.nanmean(np.where(n > 0, tp / n, np.nan), 1))
                f1.append(np.nanmean(np.where(n > 0, 2 * tp / (pc + n), np.nan), 1))
        return np.mean(ba, 0), np.mean(f1, 0)
    (bb, fb), (bj, fj) = scores(KB, QB), scores(KJ, QJ)
    return {'balanced_accuracy': bj - bb, 'macro_f1': fj - fb}

def mcnemar(y, pb, pj):
    b = int(((pb == y) & (pj != y)).sum()); c = int(((pb != y) & (pj == y)).sum()); n = b + c
    p = 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)
    return b, c, p

rows, seed_rows, recall_rows, report = [], [], [], []
say = lambda s='': (print(s), report.append(s))
for k, g in SETS:
    y, PB, PJ, art = get(k, g)
    boot = boot_painters(y, PB, PJ, art)
    for mname, fn in (('balanced_accuracy', bal_acc), ('macro_f1', macro_f1), ('top1', lambda y, p: float((y == p).mean()))):
        b = np.array([fn(y, p) for p in PB]); j = np.array([fn(y, p) for p in PJ]); d = j - b
        r = dict(set=label(k, g), metric=mname, n=len(y), baseline=100 * b.mean(), baseline_sd=100 * b.std(ddof=1),
                 colorjitter=100 * j.mean(), colorjitter_sd=100 * j.std(ddof=1), diff=100 * d.mean(),
                 t_test_p=float(stats.ttest_rel(j, b).pvalue))
        if mname in boot:
            bd = boot[mname]
            r.update(ci_low=100 * np.nanpercentile(bd, 2.5), ci_high=100 * np.nanpercentile(bd, 97.5),
                     boot_p=float(min(1.0, 2 * min((bd <= 0).mean(), (bd >= 0).mean()))))
        rows.append(r)
        for s, bs, js in zip(SEEDS, b, j):
            seed_rows.append(dict(set=label(k, g), metric=mname, seed=s, baseline=100 * bs, colorjitter=100 * js))
    for s, pb, pj in zip(SEEDS, PB, PJ):
        bb, cc, p = mcnemar(y, pb, pj)
        seed_rows.append(dict(set=label(k, g), metric='mcnemar', seed=s, only_baseline_right=bb, only_jitter_right=cc, p=p))
    for c, cname in enumerate(CLASSES):
        m = y == c
        if m.any():
            rb = 100 * np.mean([(p[m] == c).mean() for p in PB]); rj = 100 * np.mean([(p[m] == c).mean() for p in PJ])
            recall_rows.append(dict(set=label(k, g), style=cname, n=int(m.sum()), baseline=rb, colorjitter=rj, diff=rj - rb))

summary = pd.DataFrame(rows).round(3); seeds_df = pd.DataFrame(seed_rows).round(4); recall = pd.DataFrame(recall_rows).round(1)
summary.to_csv(f'{OUT}/colorjitter_summary.csv', index=False)
seeds_df.to_csv(f'{OUT}/colorjitter_per_seed.csv', index=False)
recall.to_csv(f'{OUT}/colorjitter_per_style_recall.csv', index=False)
print('Saved colorjitter_summary.csv, colorjitter_per_seed.csv, colorjitter_per_style_recall.csv to', OUT)

## 5. Results table (paste this back)

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
say('Colour jitter (brightness 0.2, saturation 0.2, training only) vs baseline ResNet-50, mean of 3 seeds, in %')
say('Museum rows marked (unseen) = painters not in WikiArt (main result). diff = jitter - baseline.')
say('CI and boot_p: painter-level paired bootstrap (2,000). t_test_p: paired t-test over the 3 seed pairs.\n')
for mname in ('balanced_accuracy', 'macro_f1', 'top1'):
    t = summary[summary.metric == mname].drop(columns='metric')
    if mname == 'top1': t = t.drop(columns=[c for c in ('ci_low', 'ci_high', 'boot_p') if c in t])
    pcols = [c for c in t.columns if c.endswith('_p')]
    t = t.round(1); t[pcols] = summary.loc[t.index, pcols].round(3)
    say(f'--- {mname} ---'); say(t.to_string(index=False)); say()
say('--- McNemar per seed (same paintings): paintings only one model gets right ---')
mc = seeds_df[seeds_df.metric == 'mcnemar'][['set', 'seed', 'only_baseline_right', 'only_jitter_right', 'p']].astype({'only_baseline_right': int, 'only_jitter_right': int})
say(mc.to_string(index=False)); say()
say('--- Per-style recall (mean of 3 seeds) ---')
for s in ('wikiart_test', 'met (unseen)', 'rijks (unseen)'):
    say(s); say(recall[recall.set == s].drop(columns='set').to_string(index=False)); say()
say(f'Check: baseline means should read WikiArt 61.1, Met all 61.0 / unseen 57.5, Rijks all 58.0 / unseen 58.2 (balanced accuracy).')
open(f'{OUT}/colorjitter_report.txt', 'w').write('\n'.join(report))
print('\nSaved colorjitter_report.txt to', OUT)